In [ ]:
import pandas as pd 
import os
import matplotlib.pyplot as plt
import matplotlib.patches
from matplotlib import cm
from IPython.display import Image, display
import numpy as np
import sys
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from math import sqrt
import math
from functools import reduce
from matplotlib import colors
from matplotlib.colors import ListedColormap
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D
import shap
import time
from datetime import date
from datetime import datetime

# Load Data

In [ ]:
def process_df(df, model, feature_code, version):
    prefix = version.upper() + '2' if version == 'gfn' else version.upper()
    
    if model == 'RFa':
        data = df.drop(columns = [
            'Cation Dihedral Angle', 'Mean Dihedral Angle_c', 'Delta Dihedral Angle_c',
            'Cation Bond Length', 'Mean Bond Length_c', 'Delta Bond Length_c', 
            f'{prefix} Cation λ'
        ]).rename(columns = {
            'Mean Dihedral Angle_a': 'Mean Dihedral Angle',
            'Delta Dihedral Angle_a': '| $\Delta$ Dihedral Angle |',
            'Mean Bond Length_a': 'Mean Bond Length',
            'Delta Bond Length_a': '| $\Delta$ Bond Length |',
            f'{prefix} Anion λ': f'{prefix} λ'
        })

    if model == 'RFc':
        data = df.drop(columns = [
            'Anion Dihedral Angle', 'Mean Dihedral Angle_a', 'Delta Dihedral Angle_a',
            'Anion Bond Length', 'Mean Bond Length_a', 'Delta Bond Length_a',
            f'{prefix} Anion λ'
        ]).rename(columns = {
            'Mean Dihedral Angle_c': 'Mean Dihedral Angle',
            'Delta Dihedral Angle_c': '| $\Delta$ Dihedral Angle |',
            'Mean Bond Length_c': 'Mean Bond Length',
            'Delta Bond Length_c': '| $\Delta$ Bond Length |',
            f'{prefix} Cation λ': f'{prefix} λ'
        })
        
    target_column = f'{prefix} λ'

    feature_abbrev_dict = {
        'psa': 'PiSystemArea',
        'psl': 'PiSystemLength',
        'pss': 'PiSystemSize',
        'angles': 'Angle',
        'lengths': 'Bond Length',
        'means': 'Mean',
        'deltas': 'Delta',
        'ecfp': 'ECFP',
        'rdkfp': 'RDKFP',
        'pi': 'Pi',
    }

    if feature_code != 'all-features':
        if feature_code == 'only-ecfp':
            data = data[['Monomer 1', 'Monomer 2', 'Oligomer Length'] + [f"ECFP_{i}" for i in range(2048)] + [target_column]]
        elif 'only' in feature_code:
            columns = ['Monomer 1', 'Monomer 2', 'Oligomer Length']
            selected_fts = feature_code.split('-')[1:]
            for fs in selected_fts:
                selected_column = feature_abbrev_dict.get(fs)
                selected_column_names = [c for c in data.columns if selected_column in c]
                columns = columns + selected_column_names
            data = data[columns + [target_column]]
        else:
            feature_selection = feature_code.split('-')[1:]
            for fs in feature_selection:
                column_name = feature_abbrev_dict.get(fs)
                data = data.drop(list(data.filter(regex = column_name)), axis = 1)

    if version == 'dft':
        data_df = data[data['DFT λ'] > 0].reset_index(drop = True)
        data_df = data_df[data_df['DFT λ'] < 2].reset_index(drop = True)
        data_df = data_df.dropna().reset_index(drop = True)
    else:
        data_df = data.drop(columns = ['GFN2 λ'])
        data_df = data_df.dropna().reset_index(drop = True)
    
    print(list(dict.fromkeys(x.split('_')[0] for x in data_df.columns)))
    return data, data_df

In [ ]:
def set_params(model, feature_code, ttrs, tts, fp):
    num_trees_dict = {
        'RFc_no-deltas-psa-psl_ecfp4': [25, 'original'],
        'RFc_no-deltas-psa-psl_ecfp6': [50, 'original'],
        'RFc_no-deltas-psa-psl_rdkfp': [25, 'original'],
    
        'RFc_no-psa-psl_ecfp4': [25, 'oldpi'],
        'RFc_no-psa-psl_ecfp6': [50, 'oldpi'],
        'RFc_no-psa-psl_rdkfp': [50, 'oldpi'],
        
        'RFa_no-deltas-psa-psl_ecfp4': [50, 'original'],
        'RFa_no-deltas-psa-psl_ecfp6': [50, 'original'],
        'RFa_no-deltas-psa-psl_rdkfp': [50, 'original'],
        
        'RFa_no-psa-psl_ecfp4': [50, 'oldpi'],
        'RFa_no-psa-psl_ecfp6': [50, 'oldpi'],
        'RFa_no-psa-psl_rdkfp': [50, 'oldpi'],
        
        'RFa_no-deltas-means-psa-psl_ecfp4': [50, 'no_deltas_means_oldpi'],
        'RFc_no-deltas-means-psa-psl_ecfp4': [50, 'no_deltas_means_oldpi'],
        
        'RFc_only-ecfp_ecfp4': [50, 'ecfp'],
        'RFa_only-ecfp_ecfp4': [50, 'ecfp'],
        
        'RFc_only-pss_ecfp4': [50, 'pss'],
        'RFa_only-pss_ecfp4': [50, 'pss'],
        
        'RFc_only-angles-lengths_ecfp4': [50, 'geos'],
        'RFa_only-angles-lengths_ecfp4': [50, 'geos'],
        
        'RFc_only-angles-lengths-ecfp_ecfp4': [50, 'no_pi'],
        'RFa_only-angles-lengths-ecfp_ecfp4': [50, 'no_pi'],
        
        'RFc_only-angles-lengths-pss_ecfp4': [50, 'no_ecfp4'],
        'RFa_only-angles-lengths-pss_ecfp4': [50, 'no_ecfp4'],
        
        'RFc_only-ecfp-pss_ecfp4': [50, 'no_geo'],
        'RFa_only-ecfp-pss_ecfp4': [50, 'no_geo'],
    }
        
    desc_str = f"{model}_{feature_code}_{fp}"
    num_trees = num_trees_dict.get(desc_str)[0]
    feature_abbrev = num_trees_dict.get(desc_str)[1]
    
    output_path = os.path.join(os.getcwd(), 'output', desc_str)
    os.makedirs(output_path, exist_ok=True)
    
    _, m, d = str(date.today()).split('-')
    mddate = m + d
    suffix = f"{feature_abbrev}_{fp}-{ttrs}-{mddate}"
    
    params = [model, feature_code, suffix, output_path, fp]
    return num_trees, params

In [ ]:
def graph_feature_importances(features, importances, fp_sum, fi_method, params, preview):
    def _default_graph(features, importances, add_fp, fp_sum, fi_method, params, preview):
        model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
        
        if color_scheme == 'cb10':
            my_cmap = ListedColormap(color_palette_list, name='cb10')
        else:
            my_cmap = plt.get_cmap(color_scheme)
            
        fi_method_name = '-SHAP' if fi_method == 'SHAP' else ''
        
        importances = np.asarray(importances)
        features = np.asarray(features)

        indices = np.argsort(importances)
        num_features = min(10, len(importances)) 
        
        fig, ax = plt.subplots(figsize=(10,6))
        
        if add_fp == True:
            custom_colors = ['#291EAB'] + [my_cmap(i / num_features) for i in reversed(range(num_features))]
            graph_filename = f"{model}_MIF{fi_method_name}_{suffix}_fpsum"
            
            ax.barh(range(num_features +1), np.append(fp_sum, importances[indices[-num_features:]]), color = custom_colors, edgecolor = 'k', align='center')
            fp_name = fp.upper()
            plt.yticks(range(num_features +1), [f'{fp_name} Bits Sum'] + [features[i] for i in indices[-num_features:]], fontsize = 16)
            
        else:
            custom_colors = [my_cmap(i / num_features) for i in reversed(range(num_features))]
            graph_filename = f"{model}_MIF{fi_method_name}_{suffix}"
            
            ax.barh(range(num_features), importances[indices[-num_features:]], color = custom_colors, edgecolor = 'k', align='center')
            plt.yticks(range(num_features), [features[i] for i in indices[-num_features:]], fontsize = 16)
        
        xlabel_name = 'Mean Absolute SHAP Value' if fi_method == 'SHAP' else 'Relative Importance' 
        plt.xlabel(xlabel_name, fontsize = 20)
        plt.xticks(fontsize = 16)
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f'{graph_filename}.pdf'))
        
        if preview == False:
            plt.close()
    
    for add_fp in [True, False]:
        _default_graph(features, importances, add_fp, fp_sum, fi_method, params, preview)


def get_feat_imp(data_df, regr, params, preview):
    def _default(data_df, regr, X, params, preview):
        model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
        
        print('Default feature importances:')
        for x,y in sorted(zip(data_df.columns, regr.feature_importances_), reverse = True, key = lambda x: x[1])[:10]:
            print(f'{x}: {y*100:.2f}%')
        print()
        
        feature_importances = sorted(zip(data_df.columns, regr.feature_importances_), reverse = True, key = lambda x: x[1])
        feature_df = pd.DataFrame(feature_importances, columns = ['Feature', 'Importance'])
        feature_df['Importance %'] = feature_df['Importance']*100
        feature_df.to_csv(os.path.join(output_path, f"{model}_FI_{suffix}.csv"), index = False)

        fp_sum = 0
        for item in sorted(zip(data_df.columns, regr.feature_importances_)):
            if 'FP_' in item[0]:
               fp_sum += item[1]
        print(f"\nSum of fingerprint importances: {fp_sum}\n")
        
        graph_feature_importances(X.columns, regr.feature_importances_, fp_sum, 'default', params, preview)
    
    
    def _shap(regr, X, params, preview):
        model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
        
        explainer = shap.Explainer(regr, X)
        shap_values = explainer(X, check_additivity=False)
        
        shap_df = pd.DataFrame(
            {
                'Feature': X.columns,
                'SHAP Value': np.abs(shap_values.values).mean(axis = 0)
            }
        ).sort_values(by = ['SHAP Value'], ascending = False, ignore_index = True)

        print('SHAP Importances')
        for index, row in shap_df.head(10).iterrows():
            print(f"{row['Feature']}: {row['SHAP Value']}")
        print()
        
        shap_df.to_csv(os.path.join(output_path, f"{model}_SHAP-FI_{suffix}.csv"), index = False)

        fp_sum = 0
        for index, row in shap_df.iterrows():
            if 'ECFP' in row['Feature']:
                fp_sum += shap_df.loc[index, 'SHAP Value']
        print(f"Sum of fingerprint importances: {fp_sum}")
        
        graph_feature_importances(shap_df['Feature'], shap_df['SHAP Value'], fp_sum, 'SHAP', params, preview)
    
    X = data_df.drop(columns = ['DFT λ'])
    
    _default(data_df, regr, X, params, preview)
    _shap(regr, X, params, preview)

In [ ]:
def graph_predictions(pdf, params, preview):
    model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
    xlim_max = pdf['DFT λ'].max()
    xlim_max_padded = math.ceil(pdf['DFT λ'].max() * 10) / 10
    
    fig=plt.figure(figsize=(8, 6), dpi= 300, facecolor='w', edgecolor='k')
    plt.grid(color='k', ls = '-.', lw = 0.1, alpha = 0.5)
    plt.xlim(0,xlim_max)
    reg_x = np.arange(0,xlim_max,0.1)
    
    if len(pdf[pdf['Oligomer Length'] == 4]) > 0:
        tetra_sct = plt.scatter(
            pdf[pdf['Oligomer Length'] == 4]['Trained λ'], 
            pdf[pdf['Oligomer Length'] == 4]['DFT λ'], 
            s = 15, c = '#648FFF',
            alpha = 0.5, zorder = 0, label = "Tetramers")
        slope, intercept, t_r_value, p_value, std_err = linregress(
            pdf[pdf['Oligomer Length'] == 4]['Trained λ'].astype(float),
            pdf[pdf['Oligomer Length'] == 4]['DFT λ'])
        tet_plt = plt.plot(reg_x,slope*reg_x + intercept, c = '#2661E7', lw = 2, alpha = 1, zorder = 1)
        t_rmse = sqrt(mean_squared_error(
            pdf[pdf['Oligomer Length'] == 4]['Trained λ'],
            pdf[pdf['Oligomer Length'] == 4]['DFT λ']))
        
    if len(pdf[pdf['Oligomer Length'] == 6]) > 0:
        hex_sct = plt.scatter(
            pdf[pdf['Oligomer Length'] == 6]['Trained λ'], 
            pdf[pdf['Oligomer Length'] == 6]['DFT λ'], 
            marker = 'X', s = 25, c = '#DC267F',
            linewidths = 0.15, alpha = 0.7, zorder = 0, label = "Hexamers")
        slope, intercept, h_r_value, p_value, std_err = linregress(
            pdf[pdf['Oligomer Length'] == 6]['Trained λ'].astype(float),
            pdf[pdf['Oligomer Length'] == 6]['DFT λ'])
        hex_plt = plt.plot(reg_x,slope*reg_x + intercept, c ='#DC267F', ls = (0, (3, 1, 1, 1)), lw = 2, alpha = 1, zorder = 1)
        h_rmse = sqrt(mean_squared_error(
            pdf[pdf['Oligomer Length'] == 6]['Trained λ'],
            pdf[pdf['Oligomer Length'] == 6]['DFT λ']))
    
    slope, intercept, total_r_value, p_value, std_err = linregress(pdf['Trained λ'].astype(float),pdf['DFT λ'])
    total_plt, = plt.plot(reg_x,slope*reg_x + intercept, c = '#FFB000', ls = (0, (3, 5, 1, 5)), lw = 2, alpha = 1, zorder = 2, label = 'Total')
    
    total_rmse = sqrt(mean_squared_error(pdf['Trained λ'],pdf['DFT λ']))
    total = matplotlib.patches.Rectangle((0,0), 1, 1, fill=False, edgecolor='none', visible=False)
    plt.xlabel('Predicted λ (eV)', fontsize = 18)
    plt.ylabel('B3LYP λ (eV)', fontsize = 18)
    plt.xticks(fontsize = 16)
    plt.yticks(fontsize = 16)
    
    plt.legend([tet_plt[0], hex_plt[0], total_plt], 
               ['Tetramers\n$R^2$ = {:.3f}\nRMSE = {:.3f} eV'.format(t_r_value ** 2, t_rmse), 
                'Hexamers\n$R^2$ = {:.3f}\nRMSE = {:.3f} eV'.format(h_r_value ** 2, h_rmse),
                'Total\n$R^2$ = {:.3f}\nRMSE = {:.3f} eV'.format(total_r_value ** 2, total_rmse),], 
               loc = 8, ncol=3, bbox_to_anchor=(0.49, -0.33), markerscale = 1.5, fontsize = 12, frameon = True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f'{model}_TP_{suffix}.pdf'))
    #plt.savefig(os.path.join(output_path, f'{model}_TP_{suffix}.png'))
    if preview == False:
        plt.close()
    


def get_predictions_all(regr, test_X, test_y, params, preview):
    model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
    
    test_y_predictions = np.asarray(regr.predict(test_X))
    pdf = test_X[['Monomer 1','Monomer 2','Oligomer Length']].copy(deep=True)
    pdf['DFT λ'] = np.asarray(test_y)
    pdf['Trained λ'] =  [x for x in list(test_y_predictions)]
    pdf['PctDiff'] = (abs(pdf['DFT λ']-pdf['Trained λ'])/pdf['DFT λ'])*100
    print('In the test set there are...')
    
    print(f"Tetramers: {len(pdf[pdf['Oligomer Length'] == 4])}")
    print(f"Hexamers: {len(pdf[pdf['Oligomer Length'] == 6])}")
    
    predicted_points = pdf[['Monomer 1', 'Monomer 2', 'Oligomer Length', 'DFT λ', 'Trained λ', 'PctDiff']]
    predicted_points.to_csv(os.path.join(output_path, f"{model}_PP_{suffix}.csv"), index = False)
    
    graph_predictions(pdf, params, preview)

In [ ]:
def graph_common_monomer_freq(common_monomers, common_mono_df, olig_name, params, preview):
    def _normal_monofreq(common_monomers, olig_name, suffix, output_path, preview):
        plt.figure(figsize=(10, 5), facecolor='w', edgecolor='k')
        n, bins, patches = plt.hist(common_monomers, bins = range(1,313,1))
        fracs = n / n.max()
        norm = colors.Normalize(fracs.min(), fracs.max())
        for thisfrac, thispatch in zip(fracs, patches):
            color = plt.cm.viridis(norm(thisfrac))
            thispatch.set_facecolor(color)

        plt.xlabel('Monomer Number', fontsize = 16)
        plt.ylabel('Frequency', fontsize = 16)
        plt.xticks(fontsize = 16)
        plt.yticks(fontsize = 16)
        plt.xlim(0,313)
        
        #plt.title('Monomer Frequency in the Best Performing Hexamers')
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f"{model}_PCM-{olig_name.upper()[0]}_{suffix}.pdf"))
        if preview == False:
            plt.close()
    
    def _desc_monofreq(common_mono_df, olig_name, suffix, output_path, preview):
        plt.figure(figsize=(10, 5), facecolor='w', edgecolor='k')
        my_cmap = plt.get_cmap(name = 'viridis_r')
        sorted_df = common_mono_df.sort_values(by = 'Count', ascending = False)
        plt.bar(range(1, 313), sorted_df['Count'].values, color = my_cmap(np.arange(0,125,0.05)))

        plt.ylabel('Frequency', fontsize = 18)
        plt.tick_params(
            axis='x',
            which='both',
            bottom=False,
            top=False,
            labelbottom=False)
        plt.yticks(fontsize = 16)
        plt.xlim(0,313)
        
        #plt.title('Monomer Frequency in the Best Performing Hexamers')
        plt.tight_layout()
        plt.savefig(os.path.join(output_path, f"{model}_PCMS-{olig_name.upper()[0]}_{suffix}.pdf"))
        if preview == False:
            plt.close()
    
    model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
    
    _normal_monofreq(common_monomers, olig_name, suffix, output_path, preview)
    _desc_monofreq(common_mono_df, olig_name, suffix, output_path, preview)




def graph_screening_predictions(new_df, olig_length, params, preview):
    bin_width = 0.005
    benchmark = 0.3
    model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
    olig_color = '#648FFF' if olig_length == 4 else '#DC267F'
    oligomer_name = 'Tetramer' if olig_length == 4 else 'Hexamer'
    
    fig=plt.figure(figsize=(10, 6), dpi= 200, facecolor='w', edgecolor='k')
    
    n, bins, patches = plt.hist(new_df['Trained λ'], bins = np.arange(min(new_df['Trained λ']), max(new_df['Trained λ']) + bin_width, bin_width), color = olig_color)
    for index, t_bin in enumerate(bins):
        if t_bin < benchmark:
            patches[index].set_fc('#FFB000')
            
    new_olig = len(new_df[new_df['Trained λ'] < benchmark])
    new_olig_pct = len(new_df[new_df['Trained λ'] < benchmark])*100 / len(new_df)
    plt.annotate(f'New {oligomer_name}s with λ < {benchmark} eV: {new_olig}', xy = (0.5,0.9), xycoords='axes fraction', color = 'k', size = 12)
    plt.annotate('% of Total: {:.2f}%'.format(new_olig_pct), xy = (0.5,0.85), xycoords='axes fraction', color = 'k', size = 12)
    #plt.title('λ Prediction for New Tetramers', fontsize = 26)
    plt.xlabel('Trained λ (eV)', fontsize = 18)
    plt.ylabel('Frequency', fontsize = 18)
    plt.xticks(fontsize = 14)
    plt.yticks(fontsize = 14)
    plt.grid(axis='y', alpha = 0.2)
    plt.tight_layout()
    plt.savefig(os.path.join(output_path, f'{model}-{olig_length}_NP_{suffix}.pdf'))
    #plt.savefig(os.path.join(output_path, f'{model}-4_NP_{suffix}.png'))
    if preview == False:
        plt.close()




def predict_all_df(data_df, num_trees, olig_length, fp, params, preview):
    model, feature_code, suffix, output_path, fp, color_palette_list, color_scheme = params
    olig_name = 'tet' if olig_length == 4 else 'hex'
    
    rf_regr = RandomForestRegressor(n_estimators = num_trees, max_features = None, min_samples_split=2, random_state=0, verbose = 0)
    
    X = data_df.drop(['DFT λ'], axis = 1)
    y = data_df['DFT λ']
    
    start_time = time.process_time()
    rf_regr.fit(X, y)
    end_time = time.process_time()
    
    print(f"Time to train regressor: {end_time - start_time} seconds")
    
    gfn_df = pd.read_csv(f'/ix/ghutchison/kwl20/reorganization/csvs/final/{olig_name}_gfn_{fp}.csv')
    _, all_df = process_df(gfn_df, model, feature_code, 'gfn')
    print(f"{all_df.shape=}")
    print(f"all_df: {list(dict.fromkeys([x.split('_')[0] for x in all_df.columns]))}\n")
    
    old_df = data_df[data_df['Oligomer Length'] == olig_length].reset_index(drop = True)
    old_df.drop(['DFT λ'], axis = 1, inplace = True)
    print(f"old_df (input df) length: {len(old_df)}")
    print(f"all_df (full df) length: {len(all_df)}\n")
    
    new_df = pd.concat([old_df, all_df], ignore_index = True, sort=False).drop_duplicates(subset = ['Monomer 1', 'Monomer 2'], keep = False).dropna(axis = 1)
    print(f"new_df length: {len(new_df)}")
    
    pt_predictions = rf_regr.predict(new_df)
    new_df['Trained λ'] = pt_predictions
    
    graph_screening_predictions(new_df, olig_length, params, preview)
    
    if model == 'RFa':
        common_df = new_df[[
            'Monomer 1', 'Monomer 2', 'Neutral Dihedral Angle', 'Anion Dihedral Angle', 'Neutral Bond Length', 'Anion Bond Length', 'Trained λ'
        ]].reset_index(drop = True)
    if model == 'RFc':
        common_df = new_df[[
            'Monomer 1', 'Monomer 2', 'Neutral Dihedral Angle', 'Cation Dihedral Angle', 'Neutral Bond Length', 'Cation Bond Length', 'Trained λ'
        ]].reset_index(drop = True)
    
    common_df = common_df.sort_values(by = ['Trained λ'])[:150]
    common_df.to_csv(os.path.join(output_path, f"{model}_C{olig_name.upper()[0]}_{suffix}.csv"), index = False)
    
    low_pts = new_df[new_df['Trained λ'] < 0.3].sort_values(by = ['Trained λ'], ascending = False)[['Monomer 1', 'Monomer 2', 'Trained λ']]
    low_pts.to_csv(os.path.join(output_path, f"{model}_H{olig_name.upper()[0]}_{suffix}.csv"), index = False)
    
    common_monomers = list(common_df['Monomer 1'].astype(int)) + list(common_df['Monomer 2'].astype(int))
    common_mono_df = pd.DataFrame(columns = ['Monomer', 'Count'])
    rows = []
    
    for i in range(1, 313):
        count = common_monomers.count(i)
        rows.append({'Monomer': i, 'Count': count})
    
    common_mono_df = pd.DataFrame(rows, columns = ['Monomer', 'Count'])
    cm_df = common_mono_df.sort_values(by = ['Count'], ascending = False)
    cm_df.to_csv(os.path.join(output_path, f"{model}_CM-{olig_name}_{suffix}.csv"), index = False)
    graph_common_monomer_freq(common_monomers, common_mono_df, olig_name, params, preview)

In [ ]:
def main(model, feature_code, fp, ttrs, preview):
    color_palette_10 = ['#24E4BC', '#00CFF9', '#648FFF', '#785EF0', '#AA42B8', '#DC267F', '#ED2F2C', '#FE6100', '#FFB000', '#FFE200']
    color_palette_5 = ['#648FFF', '#785EF0', '#DC267F', '#FE6100', '#FFB000']
    input_file = f"/ix/ghutchison/kwl20/reorganization/csvs/final/gfn_geo_dft_energy_{fp}.csv"
    
    num_trees, init_params = set_params(model, feature_code, ttrs, 15, fp)
    params = init_params + [color_palette_10, 'cb10']
    print('Params:')
    for x in range(len(params)):
        print(params[x])
    print()
    
    df = pd.read_csv(input_file)
    data_df = df.copy()
    data, data_df = process_df(data_df, model, feature_code, 'dft')
    print(f"{data_df.shape=}")
    
    
    X = data_df.drop(['DFT λ'], axis = 1)
    print(f"{X.shape=}")
    y = data_df['DFT λ']

    train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.15, random_state=ttrs)
    n_cols = len(X.columns)
    
    regr = RandomForestRegressor(n_estimators = num_trees, max_features = None, min_samples_split=2, random_state=0, verbose = 0)
    
    start_time = time.time()
    regr.fit(train_X, train_y)
    end_time = time.time()
    
    print(f"Time to train regressor: {end_time - start_time} seconds")
    print(f"Regressor score: {regr.score(test_X, test_y)}\n")
    
    get_feat_imp(data_df, regr, params, preview)
    get_predictions_all(regr, test_X, test_y, params, preview)
    
    if 'only' not in feature_code:
        for olig_length in [4, 6]:
            predict_all_df(data_df, num_trees, olig_length, fp, params, preview)

In [ ]:
ttrs_vals = [0, 1, 42, 420, 4937548]
feature_codes = ['no-deltas-means-psa-psl']

for feature_code in feature_codes:
    for model in ['RFc', 'RFa']:
        for fp in ['ecfp4']:
            for ttrs in ttrs_vals:
                main(model, feature_code, fp, ttrs, preview = False)